# Inferring
# 推断
In this lesson, you will infer sentiment and topics from product reviews and news articles.
在本课程中，您将从产品评论和新闻文章中推断情绪和主题。

## Setup
## 设置

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

load_dotenv(find_dotenv())

# 创建客户端（指向通义千问 API）
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

def get_completion(prompt, model="qwen-max"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

**Note**: In June 2023, OpenAI updated gpt-3.5-turbo. The results you see in the notebook may be slightly different than those in the video. Some of the prompts have also been slightly modified to product the desired results.
**注**：2023 年 6 月，OpenAI 更新了 gpt-3.5-turbo。您在笔记本中看到的结果可能与视频中的结果略有不同。一些提示也进行了轻微修改，以产生所需的结果。

## Product review text
## 产品评论文字

下方内容中文：我的卧室需要一盏漂亮的灯，这盏灯有额外的储物空间，而且价格也不算太高。很快就收到了。我们灯的灯串在运输途中断了，公司很乐意地给我们寄了一条新的。几天之内就到了。组装起来很容易。我缺了一个零件，所以我联系了他们的客服，他们很快就帮我找到了！Lumina 在我看来是一家很棒的公司，关心他们的客户和产品！

In [2]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)
## 情绪（正面/负面）

下方提示词中文

以下以三重反引号分隔的产品评论的情绪是什么？

评论文本：'''{lamp_review}'''

In [3]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the provided product review is **positive**. The reviewer expresses satisfaction with the lamp's features, price, and fast delivery. They also highlight the company's good customer service, which promptly resolved issues with a broken part and a missing piece. The review concludes with praise for the company, indicating that it cares about its customers and products.


In [4]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

positive


## Identify types of emotions
## 识别情绪类型

下方提示词中文

列出以下评论作者表达的情感。最多包含五项。答案请以小写字母开头，并用逗号分隔。

评论文本：'''{lamp_review}'''

In [5]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

satisfaction, relief, happiness, appreciation, contentment


## Identify anger
下方提示词中文

以下评论的作者是否表达了愤怒？该评论以三个反引号分隔。请回答“是”或“否”。

评论文本：'''{lamp_review}'''

In [6]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews
## 从客户评论中提取产品和公司名称

下方提示词中文

从评论文本中识别以下内容：
- 评论者购买的商品
- 生产该产品的公司

评论以三个反引号分隔。请将您的回复格式化为 JSON 对象，并以“商品”和“品牌”作为键。
如果信息不存在，则使用“未知”作为值。
让你的回答尽可能简短。

评论文本：'''{lamp_review}'''

In [7]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
  "Item": "lamp",
  "Brand": "Lumina"
}


## Doing multiple tasks at once
## 一次执行多个任务

下方提示词中文

从评论文本中识别以下内容：
- 情绪（积极或消极）
- 评论者是否表达了愤怒？（正确或错误）
- 评论者购买的商品
- 生产该产品的公司

评论以三个反引号分隔。请将您的回复格式化为 JSON 对象，并以“情绪”、“愤怒”、“商品”和“品牌”作为键。
将愤怒值格式化为布尔值。
如果信息不存在，则使用“未知”作为值。
让你的回答尽可能简短。

评论文本：'''{lamp_review}'''

In [8]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "lamp",
  "Brand": "Lumina"
}


## Inferring topics
## 推断主题

下方内容中文

在政府最近进行的一项调查中，公共部门员工被要求评价他们对其所在部门的满意度。结果显示，NASA 是最受欢迎的部门，满意度高达 95%。

NASA 员工约翰·史密斯 (John Smith) 对调查结果表示：“NASA 的胜出并不让我感到惊讶。这里是一个很棒的工作场所，拥有优秀的人才和难得的机会。我很自豪能成为这样一个富有创新精神的组织的一员。”

这一结果也得到了 NASA 管理团队的欢迎，局长汤姆·约翰逊表示：“我们很高兴听到员工对在 NASA 的工作感到满意。我们拥有一支才华横溢、敬业奉献的团队，他们孜孜不倦地致力于实现我们的目标，看到他们的辛勤工作得到回报，我们感到非常高兴。”

调查还显示，社会保障局的员工满意度最低，仅有 45%的员工表示对工作感到满意。政府已承诺解决员工在调查中提出的问题，并努力提高所有部门的员工工作满意度。

In [9]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

## Infer 5 topics
## 推断5个主题

下方提示词中文

确定以下文本中要讨论的五个主题，以三重反引号分隔。

使每个项目的长度为一到两个单词。

将您的回复格式化为以逗号分隔的列表。

文本示例：'''{story}'''

In [10]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

survey, satisfaction, NASA, employees, improvement


In [11]:
response.split(sep=',')

['survey', ' satisfaction', ' NASA', ' employees', ' improvement']

In [12]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

## Make a news alert for certain topics
## 针对某些主题发出新闻提醒

下方提示词中文

确定以下主题列表中的每个项目是否是下面文本中的主题，该主题以三重反引号分隔。

请给出如下答案：
列表中的项目：0 或 1

主题列表：{", ".join(topic_list)}

文本示例：'''{story}'''

注意原来Give your answer as follows，被改成Give your answer **exactly**  as follows, with no additional text, explanations, or formatting:中文大模型喜欢画蛇添足，有时候需要这些说明


In [16]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer **exactly**  as follows, with no additional text, explanations, or formatting:
item from the list: 0 or 1
生成内容格式为JSON，topic和得出的答案result为JSON的key
List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

{
  "nasa": 1,
  "local government": 0,
  "engineering": 0,
  "employee satisfaction": 1,
  "federal government": 1
}


In [14]:
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep='\n')}
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

ALERT: New NASA story!


## Try experimenting on your own!
## 尝试自己尝试一下！